In [37]:
import numpy as np
import pandas as pd
import os
import pyarrow as pa
from scipy import sparse
from pathlib import Path
TMP_DIR = Path("./tmp")
TMP_DIR.mkdir(parents=True, exist_ok=True)

In [19]:
# --- Step 1: 轻量检查数据的列、行数、示例行（不会把整表载入内存） ---

# 路径
csv_path = Path("./data/metadata_merged.csv")
assert csv_path.exists(), f"找不到文件: {csv_path.resolve()}"

# 仅读取表头，获取列名（极快）
header_only = pd.read_csv(csv_path, nrows=0)
cols = list(header_only.columns)

print("\n[INFO] 文件路径:", csv_path.resolve())
print("[INFO] 列名（共 %d 列）:" % len(cols))
print(cols)

# 读取前几行样例，便于肉眼核验字段格式（避免一次性读全表）
sample_n = 5
sample_df = pd.read_csv(csv_path, nrows=sample_n)
print(f"\n[INFO] 前 {sample_n} 行样例：")
print(sample_df.head(sample_n))

# 简单推断：尝试读取少量行做 dtype 预览（避免低内存分块带来的混淆）
preview_n = 1000
preview_df = pd.read_csv(csv_path, nrows=preview_n, low_memory=False)
print(f"\n[INFO] 基于前 {preview_n} 行的 dtype 预览：")
print(preview_df.dtypes)

# 可选：如果数据很大，不建议现在计算完整内存；这里仅估算前1000行的内存占用
print(f"\n[INFO] 前 {preview_n} 行内存占用（近似）：")
print(preview_df.memory_usage(deep=True).sum(), "bytes")



[INFO] 文件路径: /Users/koyo/workspace/recsys/data/metadata_merged.csv
[INFO] 列名（共 31 列）:
['Id', 'Title', 'Slug', 'Tags', 'CreatorUserId', 'OwnerUserId', 'OwnerOrganizationId', 'CurrentDatasetVersionId', 'CurrentDatasourceVersionId', 'ForumId', 'Type', 'CreationDate', 'LastActivityDate', 'TotalViews', 'TotalDownloads', 'TotalVotes', 'TotalKernels', 'Medal', 'MedalAwardDate', 'Subtitle', 'Description', 'CreationDate_dt', 'LastActivityDate_dt', 'age_days', 'days_since_last_activity', 'active_30d', 'has_tags', 'TotalViews_log1p', 'TotalDownloads_log1p', 'TotalVotes_log1p', 'TotalKernels_log1p']

[INFO] 前 5 行样例：
   Id                            Title                            Slug  \
0   6   2013 American Community Survey  2013-american-community-survey   
1   7         May 2015 Reddit Comments        reddit-comments-may-2015   
2   8  Ocean Ship Logbooks (1750-1850)   climate-data-from-ocean-ships   
3   9                      Meta Kaggle                     meta-kaggle   
4  10         Hil

In [27]:
# --- Step 2 核心字段清洗 ---

import re
from pathlib import Path
import pandas as pd
import numpy as np

# 路径
csv_path = Path("./data/metadata_merged.csv")
assert csv_path.exists(), f"找不到文件: {csv_path.resolve()}"

# 仅读取本步需要的列；缺的列用空串占位
need_cols = ["Id","Title","Subtitle","Description","Tags",
             "CreationDate","LastActivityDate","CreationDate_dt","LastActivityDate_dt"]
cols_avail = list(pd.read_csv(csv_path, nrows=0).columns)
usecols = [c for c in need_cols if c in cols_avail]

df = pd.read_csv(csv_path, usecols=usecols, low_memory=False)

# 缺失填空串
for c in ["Title","Subtitle","Description","Tags"]:
    if c not in df.columns:
        df[c] = ""
    else:
        df[c] = df[c].fillna("")

# ---- Tags: 逗号切分 → 小写 → 去重（保序） ----
def _split_tags(s):
    parts = [p.strip().lower() for p in str(s).split(",") if p.strip()]
    seen, out = set(), []
    for p in parts:
        p = re.sub(r"\s+", " ", p)
        if p and p not in seen:
            out.append(p); seen.add(p)
    return out

df["tag_list"] = df["Tags"].map(_split_tags)

# ---- Text: 逐行清洗 Title + Subtitle + Description ----
url_pat = re.compile(r"https?://\S+|www\.\S+")
md_pat  = re.compile(r"\[([^\]]+)\]\([^)]+\)")

def _clean_text_row(row):
    s = " ".join([str(row.get("Title","")), str(row.get("Subtitle","")), str(row.get("Description",""))])
    s = s.replace("\n"," ").replace("\r"," ")
    s = url_pat.sub(" ", s)
    s = md_pat.sub(r"\1", s)
    s = re.sub(r"[^\w\s\-#/@&]+", " ", s)   # 保留少量符号
    s = re.sub(r"\s+", " ", s).strip().lower()
    return s

df["text_all"] = df.apply(_clean_text_row, axis=1)

# ---- 时间字段：优先 *_dt，回退到原始字符串列 ----
def _coalesce_datetime(primary, fallback):
    if primary in df.columns:
        p = pd.to_datetime(df[primary], errors="coerce")
        if fallback in df.columns:
            return p.fillna(pd.to_datetime(df[fallback], errors="coerce"))
        return p
    else:
        return pd.to_datetime(df.get(fallback, pd.Series([None]*len(df))), errors="coerce")

df["created_at"]     = _coalesce_datetime("CreationDate_dt", "CreationDate")
df["last_active_at"] = _coalesce_datetime("LastActivityDate_dt", "LastActivityDate")

# ---- 去重与编号 ----
df = df.drop_duplicates(subset=["Id"]).reset_index(drop=True)
df["doc_idx"] = np.arange(len(df), dtype=np.int64)

# 形成核心文档表（用于后续所有视图）
doc_df = df[["doc_idx","Id","text_all","tag_list","created_at","last_active_at"]].copy()

# （可选）便于后续映射：内存里保留一个索引映射 DataFrame / 字典
index_map = doc_df[["doc_idx","Id"]].copy()
id2idx = dict(zip(index_map["Id"].tolist(), index_map["doc_idx"].tolist()))

# ---- 基本统计与样例 ----
n_docs   = len(doc_df)
has_tags = (doc_df["tag_list"].str.len() > 0).sum()
avg_tags = doc_df.loc[doc_df["tag_list"].str.len() > 0, "tag_list"].map(len).mean() if has_tags>0 else 0.0
avg_text = int(doc_df["text_all"].map(lambda s: len(str(s).split())).mean())


# —— 保存到 ./tmp 供后续步骤复用 ——
doc_df.to_parquet(TMP_DIR / "doc_clean.parquet", engine="fastparquet", index=False)
index_map.to_parquet(TMP_DIR / "index_map.parquet", engine="fastparquet", index=False)

print("[INFO] 文档数:", n_docs)
print("[INFO] 至少1个标签的文档数/占比: {}/{} ({:.1f}%)".format(has_tags, n_docs, 100*has_tags/max(n_docs,1)))
print("[INFO] 平均标签数(有标签样本): {:.2f}".format(avg_tags))
print("[INFO] 平均文本词数(粗略):", avg_text)

print("\n[INFO] 样例行（前3）：")
print(doc_df.head(3).to_string(index=False))

# - doc_df: 后续视图构建的唯一真源（包含 text_all / tag_list / 时间 / doc_idx）
# - index_map / id2idx: 便于回溯 Id 与矩阵索引的对应

[INFO] 文档数: 521735
[INFO] 至少1个标签的文档数/占比: 214603/521735 (41.1%)
[INFO] 平均标签数(有标签样本): 2.08
[INFO] 平均文本词数(粗略): 38

[INFO] 样例行（前3）：
 doc_idx  Id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            

In [29]:
# --- Step 3: Tag 视图（D–T）构筑：vocab/CSR + TF-IDF + PPMI（全在内存） ---

import numpy as np
from collections import Counter
from scipy import sparse

# 先决条件：上一单元生成的 doc_df 在内存中，包含列：
# - 'doc_idx'（0..N-1）、'tag_list'（list[str]）
assert 'doc_df' in globals(), "未发现 doc_df，请先运行清洗步骤（Step 2 in-memory）。"

# ========== 可调参数 ==========
MIN_DF   = 10      # 过滤低文档频率的标签（可调：5/10/20）
MAX_VOCAB = None   # 可选限制词表上限（如 100_000），默认不过滤
ROW_NORM = True    # 是否对加权矩阵做逐行 L2 归一
# ============================

N = len(doc_df)

# 1) 统计标签的文档频率（DF）
tag_df_counter = Counter()
for tags in doc_df['tag_list']:
    if isinstance(tags, list) and tags:
        tag_df_counter.update(set(tags))   # DF（按文档去重）

raw_vocab_size = len(tag_df_counter)
kept = [(t, c) for t, c in tag_df_counter.items() if c >= MIN_DF]
kept.sort(key=lambda x: (-x[1], x[0]))
if MAX_VOCAB is not None:
    kept = kept[:MAX_VOCAB]

tag2id = {t:i for i,(t,_) in enumerate(kept)}
id2tag = [t for t,_ in kept]
df_counts = np.array([c for _,c in kept], dtype=np.int64)

pd.DataFrame({
    "tid": np.arange(len(id2tag), dtype=np.int32),
    "tag": id2tag,
    "df":  df_counts
}).to_parquet(TMP_DIR / "tag_vocab.parquet", engine="fastparquet", index=False)


V = len(tag2id)

print(f"[TAG] docs={N:,}, raw_unique_tags={raw_vocab_size:,} -> kept(min_df>={MIN_DF})={V:,}")

# 2) 构建 D–T（binary）CSR
rows, cols, data = [], [], []
for r, tags in enumerate(doc_df['tag_list']):
    if not tags: 
        continue
    for t in set(tags):
        j = tag2id.get(t)
        if j is not None:
            rows.append(r); cols.append(j); data.append(1)

if not data:
    raise RuntimeError("没有标签通过 min_df 过滤；请降低 MIN_DF。")

rows = np.asarray(rows, dtype=np.int64)
cols = np.asarray(cols, dtype=np.int64)
data = np.asarray(data, dtype=np.float32)

DT_bin = sparse.csr_matrix((data, (rows, cols)), shape=(N, V))
nnz = DT_bin.nnz
density = nnz / (N * V) if V > 0 else 0.0
docs_with_kept_tags = int((DT_bin.getnnz(axis=1) > 0).sum())

print(f"[TAG] D–T nnz={nnz:,}, density={density:.6e}, docs_with_kept_tags={docs_with_kept_tags:,} ({docs_with_kept_tags/N:.1%})")
print("[TAG] top-15 tags by df:")
topk = min(15, V)
top_idx = np.argsort(-df_counts)[:topk]
for idx in top_idx:
    print(f"   {id2tag[idx]:<30s} df={df_counts[idx]}")

# 3) TF-IDF 加权（平滑 idf）
df_vec = np.asarray(DT_bin.astype(bool).sum(axis=0)).ravel().astype(np.int64)
idf = np.log((1 + N) / (1 + df_vec)) + 1.0   # smooth idf
tfidf_data = idf[cols].astype(np.float32)    # tf=1（标签去重后为 binary）
DT_tfidf = sparse.csr_matrix((tfidf_data, (rows, cols)), shape=(N, V))

if ROW_NORM:
    row_norms = np.sqrt(np.asarray(DT_tfidf.power(2).sum(axis=1)).ravel()) + 1e-12
    DT_tfidf = DT_tfidf.multiply(1.0 / row_norms[:, None])

print(f"[TAG] TF-IDF done. row_norm check (min/median/max): "
      f"{float(np.min(np.asarray(DT_tfidf.power(2).sum(axis=1)))):.4f}/"
      f"{float(np.median(np.asarray(DT_tfidf.power(2).sum(axis=1)))):.4f}/"
      f"{float(np.max(np.asarray(DT_tfidf.power(2).sum(axis=1)))):.4f}")

# 4) PPMI 加权（正点互信息）
# PMI(d,t)=log((x_dt * Npairs)/(row_sum_d * col_sum_t)), x_dt=1；PPMI=max(PMI,0)
row_sum = np.asarray(DT_bin.sum(axis=1)).ravel().astype(np.float64)  # 每文档标签数
col_sum = np.asarray(DT_bin.sum(axis=0)).ravel().astype(np.float64)  # 每标签 DF
Npairs  = float(nnz)

num = np.ones_like(data, dtype=np.float64) * Npairs
den = row_sum[rows] * col_sum[cols]
ppmi_vals = np.maximum(0.0, np.log(num / (den + 1e-12))).astype(np.float32)
DT_ppmi = sparse.csr_matrix((ppmi_vals, (rows, cols)), shape=(N, V))

if ROW_NORM:
    ppmi_row_norms = np.sqrt(np.asarray(DT_ppmi.power(2).sum(axis=1)).ravel()) + 1e-12
    DT_ppmi = DT_ppmi.multiply(1.0 / ppmi_row_norms[:, None])

print(f"[TAG] PPMI done. nnz={DT_ppmi.nnz:,}, ppmi(mean/std)={ppmi_vals.mean():.4f}/{ppmi_vals.std():.4f}")

def _save_csr_as_parquet_triplets(mat, path):
    coo = mat.tocoo()
    pd.DataFrame({
        "row": coo.row.astype(np.int32),
        "col": coo.col.astype(np.int32),
        "val": coo.data.astype(np.float32),
    }).to_parquet(path, engine="fastparquet", index=False)

_save_csr_as_parquet_triplets(DT_bin,   TMP_DIR / "DT_bin.parquet")
_save_csr_as_parquet_triplets(DT_tfidf, TMP_DIR / "DT_tfidf.parquet")
_save_csr_as_parquet_triplets(DT_ppmi,  TMP_DIR / "DT_ppmi.parquet")

# - tag2id / id2tag / df_counts
# - DT_bin（binary），DT_tfidf（行归一），DT_ppmi（行归一）


[TAG] docs=521,735, raw_unique_tags=597 -> kept(min_df>=10)=394
[TAG] D–T nnz=445,426, density=2.166853e-03, docs_with_kept_tags=214,585 (41.1%)
[TAG] top-15 tags by df:
   pre-trained model              df=30498
   business                       df=27014
   earth and nature               df=17261
   computer science               df=12007
   arts and entertainment         df=11738
   tabular                        df=11589
   education                      df=8809
   data analytics                 df=8283
   image                          df=8064
   beginner                       df=8030
   health                         df=7581
   text                           df=7424
   data visualization             df=7388
   movies and tv shows            df=6886
   finance                        df=5731
[TAG] TF-IDF done. row_norm check (min/median/max): 0.0000/0.0000/1.0000
[TAG] PPMI done. nnz=445,426, ppmi(mean/std)=3.7901/1.3509


In [ ]:
# --- Step 4: Text 视图（D–W）构筑：BM25 加权 CSR（内存态） ---

import re
import numpy as np
from collections import Counter
from scipy import sparse

# 先决条件：doc_df 在内存，含 ['doc_idx','text_all']
assert 'doc_df' in globals(), "未发现 doc_df，请先运行清洗步骤。"

# ========== 可调参数 ==========
MIN_DF_W       = 200        # 词项最小文档频率阈值（可按机器内存调大：200/300/500）
MAX_DF_RATIO_W = 0.50       # 过滤过于常见的词（>50% 文档出现的词去掉）
MAX_VOCAB_W    = 100_000    # 词表上限（None 表示不限；建议 50k~150k）
TOKEN_MIN_LEN  = 2          # 最小 token 长度
K1, B          = 1.2, 0.75  # BM25 参数
ROW_NORM       = True       # 是否对行做 L2 归一（便于后续相似度）
PROGRESS_EVERY = 50_000     # 进度打印步长
# ============================

N = len(doc_df)
token_pat = re.compile(r"[a-z0-9]+")

def tokenize(s):
    # text_all 已经 lower；只取字母数字序列，过滤过短 token
    toks = token_pat.findall(str(s))
    return [t for t in toks if len(t) >= TOKEN_MIN_LEN]

# ---------- Pass 1: 统计文档频率（DF） ----------
df_counter = Counter()
for i, txt in enumerate(doc_df['text_all'].values):
    if i % PROGRESS_EVERY == 0 and i > 0:
        print(f"[TEXT:DF] processed {i:,}/{N:,}")
    toks = tokenize(txt)
    if not toks:
        continue
    uniq = set(toks)
    df_counter.update(uniq)

raw_vocab = len(df_counter)
max_df_abs = int(MAX_DF_RATIO_W * N)

kept = [(t, c) for t, c in df_counter.items() if c >= MIN_DF_W and c <= max_df_abs]
kept.sort(key=lambda x: (-x[1], x[0]))
if MAX_VOCAB_W is not None:
    kept = kept[:MAX_VOCAB_W]

word2id = {w:i for i,(w,_) in enumerate(kept)}
id2word = [w for w,_ in kept]
df_counts_w = np.array([c for _,c in kept], dtype=np.int64)

pd.DataFrame({
    "wid": np.arange(len(id2word), dtype=np.int32),
    "word": id2word,
    "df": df_counts_w
}).to_parquet(TMP_DIR / "text_vocab.parquet", engine="fastparquet", index=False)

V = len(word2id)

print(f"[TEXT] docs={N:,}, raw_vocab={raw_vocab:,} -> kept(min_df>={MIN_DF_W}, max_df<={MAX_DF_RATIO_W:.0%})={V:,}")

# ---------- BM25: 需要先得到每篇文档的长度（基于“保留的词”） ----------
doc_len = np.zeros(N, dtype=np.int32)
for i, txt in enumerate(doc_df['text_all'].values):
    if i % PROGRESS_EVERY == 0 and i > 0:
        print(f"[TEXT:LEN] processed {i:,}/{N:,}")
    toks = tokenize(txt)
    if not toks:
        continue
    # 仅保留词表里的词
    cnt = 0
    for t in toks:
        if t in word2id:
            cnt += 1
    doc_len[i] = cnt

avg_len = float(doc_len.mean() if N > 0 else 0.0)
print(f"[TEXT] avg_doc_len (kept-vocab based) = {avg_len:.2f}")

# ---------- 计算 idf（BM25 版本，clamp>=0） ----------
# idf = log( (N - df + 0.5) / (df + 0.5) )
idf = np.log((N - df_counts_w + 0.5) / (df_counts_w + 0.5))
idf = np.maximum(idf, 0.0).astype(np.float32)

# ---------- Pass 2: 构建 D–W 的 BM25 加权 CSR ----------
rows, cols, data = [], [], []
for i, txt in enumerate(doc_df['text_all'].values):
    if i % PROGRESS_EVERY == 0 and i > 0:
        print(f"[TEXT:CSR] processed {i:,}/{N:,}")
    toks = tokenize(txt)
    if not toks:
        continue
    # 统计该文档的 TF（仅对保留词）
    tf_local = {}
    for t in toks:
        j = word2id.get(t)
        if j is not None:
            tf_local[j] = tf_local.get(j, 0) + 1

    if not tf_local:
        continue

    Ld = max(doc_len[i], 1)
    norm = (1 - B) + B * (Ld / max(avg_len, 1e-6))

    for j, tf in tf_local.items():
        w = idf[j] * (tf * (K1 + 1)) / (tf + K1 * norm)
        if w <= 0:
            continue
        rows.append(i); cols.append(j); data.append(w)

if not data:
    raise RuntimeError("D–W 矩阵为空；请降低 MIN_DF_W 或提高 MAX_DF_RATIO_W。")

rows = np.asarray(rows, dtype=np.int64)
cols = np.asarray(cols, dtype=np.int64)
data = np.asarray(data, dtype=np.float32)

DW_bm25 = sparse.csr_matrix((data, (rows, cols)), shape=(N, V))

if ROW_NORM:
    row_norms = np.sqrt(np.asarray(DW_bm25.power(2).sum(axis=1)).ravel()) + 1e-12
    DW_bm25 = DW_bm25.multiply(1.0 / row_norms[:, None])

nnz = DW_bm25.nnz
density = nnz / (N * V) if V > 0 else 0.0
docs_with_words = int((DW_bm25.getnnz(axis=1) > 0).sum())

print(f"[TEXT] D–W nnz={nnz:,}, density={density:.6e}, docs_with_words={docs_with_words:,} ({docs_with_words/N:.1%})")
print("[TEXT] top-15 words by df:")
topk = min(15, V)
top_idx = np.argsort(-df_counts_w)[:topk]
for idx in top_idx:
    print(f"   {id2word[idx]:<20s} df={df_counts_w[idx]}")

print(f"[TEXT] BM25 done. row_norm check (min/median/max): "
      f"{float(np.min(np.asarray(DW_bm25.power(2).sum(axis=1)))):.4f}/"
      f"{float(np.median(np.asarray(DW_bm25.power(2).sum(axis=1)))):.4f}/"
      f"{float(np.max(np.asarray(DW_bm25.power(2).sum(axis=1)))):.4f}")


def _save_csr_as_parquet_triplets(mat, path):
    coo = mat.tocoo()
    pd.DataFrame({
        "row": coo.row.astype(np.int32),
        "col": coo.col.astype(np.int32),
        "val": coo.data.astype(np.float32),
    }).to_parquet(path, engine="fastparquet", index=False)

_save_csr_as_parquet_triplets(DW_bm25, TMP_DIR / "DW_bm25.parquet")

[TEXT:DF] processed 50,000/521,735
[TEXT:DF] processed 100,000/521,735
[TEXT:DF] processed 150,000/521,735
[TEXT:DF] processed 200,000/521,735
[TEXT:DF] processed 250,000/521,735
[TEXT:DF] processed 300,000/521,735
[TEXT:DF] processed 350,000/521,735
[TEXT:DF] processed 400,000/521,735
[TEXT:DF] processed 450,000/521,735
[TEXT:DF] processed 500,000/521,735
[TEXT] docs=521,735, raw_vocab=372,377 -> kept(min_df>=200, max_df<=50%)=5,714
[TEXT:LEN] processed 50,000/521,735
[TEXT:LEN] processed 100,000/521,735
[TEXT:LEN] processed 150,000/521,735
[TEXT:LEN] processed 200,000/521,735
[TEXT:LEN] processed 250,000/521,735
[TEXT:LEN] processed 300,000/521,735
[TEXT:LEN] processed 350,000/521,735
[TEXT:LEN] processed 400,000/521,735
[TEXT:LEN] processed 450,000/521,735
[TEXT:LEN] processed 500,000/521,735
[TEXT] avg_doc_len (kept-vocab based) = 32.77
[TEXT:CSR] processed 50,000/521,735
[TEXT:CSR] processed 100,000/521,735
[TEXT:CSR] processed 150,000/521,735
[TEXT:CSR] processed 200,000/521,735


In [34]:
# —— 随机游走（D-only语料） ——
RW_WALKS_PER_DOC   = 10       # 每个起点D产生的游走条数（建议10）
RW_L_DOCS_PER_SENT = 40       # D-only序列长度（建议40）
RW_SEED_BASE       = 2025     # 基础随机种子（每次__iter__会+epoch_id）
RW_AVOID_BACKTRACK = True     # 禁止立即回到上一个D
RW_RESTART_PROB    = 0.15     # 重启概率：以 d0 为“个性化”重启（PPR风格）
RW_X_DEGREE_POW    = -0.5     # X侧度数惩罚：乘以 (deg(X))^pow ，-0.5 ~ -1 有效抑制hub
RW_X_NO_REPEAT_LAST= 1        # 最近多少步内访问过的X在下一步禁用（1表示不重复上一个X）
RW_PROGRESS_EVERY  = 50_000   # 起点进度打印（适度即可）

# 仅抽样检查（不消耗太多）开关
RW_INSPECT_SAMPLES  = 0        # >0 时抽样打印若干条 D–X–D 与 D-only（高性能下建议 0）

In [35]:
# --- Step 5 · Execute: Build On-the-fly D-only Walk Corpora from Bipartite Graphs ---


doc_df = pd.read_parquet(TMP_DIR / "doc_clean.parquet", engine="fastparquet")
tag_vocab = pd.read_parquet(TMP_DIR / "tag_vocab.parquet", engine="fastparquet")
text_vocab = pd.read_parquet(TMP_DIR / "text_vocab.parquet", engine="fastparquet")

def _load_csr_from_parquet_triplets(path, shape, dtype=np.float32):
    df = pd.read_parquet(path, engine="fastparquet")
    coo = sparse.coo_matrix((df["val"].astype(dtype), (df["row"], df["col"])),
                            shape=shape, dtype=dtype)
    return coo.tocsr()

DT_ppmi = _load_csr_from_parquet_triplets("./tmp/DT_ppmi.parquet",
                                          shape=(len(doc_df), len(tag_vocab)))
DW_bm25 = _load_csr_from_parquet_triplets("./tmp/DW_bm25.parquet",
                                          shape=(len(doc_df), len(text_vocab)))

# 统一 CSR/float32
DT_ppmi = DT_ppmi.tocsr().astype(np.float32, copy=False)
DW_bm25 = DW_bm25.tocsr().astype(np.float32, copy=False)

# 预计算
degD_tag = np.asarray(DT_ppmi.getnnz(axis=1)).ravel()
degX_tag = np.asarray(DT_ppmi.getnnz(axis=0)).ravel()
degD_txt = np.asarray(DW_bm25.getnnz(axis=1)).ravel()
degX_txt = np.asarray(DW_bm25.getnnz(axis=0)).ravel()
XD_tag = DT_ppmi.transpose().tocsr()
XD_txt = DW_bm25.transpose().tocsr()
start_tag = np.where(degD_tag > 0)[0]
start_txt = np.where(degD_txt > 0)[0]

def _row_neighbors_csr(mat_csr, r):
    a,b = mat_csr.indptr[r], mat_csr.indptr[r+1]
    if a==b: return None, None
    return mat_csr.indices[a:b], mat_csr.data[a:b]

def _sample_pos_by_weights(weights, rng):
    if weights is None or len(weights)==0: return None
    w = np.asarray(weights, dtype=np.float64); tot = w.sum()
    if not np.isfinite(tot) or tot<=0: return None
    cdf = np.cumsum(w)
    pos = int(np.searchsorted(cdf, rng.random()*tot, side="left"))
    if pos >= cdf.size: pos = cdf.size-1
    return pos

class WalkCorpus:
    """按需生成的 D-only 语料；每次 __iter__ 都会重掷随机数并打乱起点。"""
    def __init__(self, DX, XD, degX, starts, name, base_seed):
        self.DX, self.XD, self.degX = DX, XD, degX
        self.starts = starts
        self.name = name
        self.base_seed = base_seed
        self._iters = 0
    def __len__(self):
        return int(len(self.starts) * RW_WALKS_PER_DOC)
    def __iter__(self):
        rng = np.random.default_rng(self.base_seed + 17*self._iters); self._iters += 1
        starts = self.starts.copy(); rng.shuffle(starts)
        x_deg_factor = np.power(np.maximum(self.degX,1.0), RW_X_DEGREE_POW).astype(np.float64) if abs(RW_X_DEGREE_POW)>1e-12 else None
        for si, d0 in enumerate(starts, 1):
            if RW_PROGRESS_EVERY and si % RW_PROGRESS_EVERY == 0:
                print(f"[RW-{self.name}] progressed: {si:,}/{len(starts):,}")
            for _ in range(RW_WALKS_PER_DOC):
                seq = [int(d0)]; prev_d=None; cur_d=int(d0); last_x=[]
                for _step in range(RW_L_DOCS_PER_SENT-1):
                    x_cols, x_w = _row_neighbors_csr(self.DX, cur_d)
                    if x_cols is None: break
                    w = x_w.astype(np.float64); 
                    if x_deg_factor is not None: w = w * x_deg_factor[x_cols]
                    if RW_X_NO_REPEAT_LAST>0 and last_x:
                        mask = np.isin(x_cols, last_x[-RW_X_NO_REPEAT_LAST:])
                        if mask.any(): w = w.copy(); w[mask]=0.0
                    pos_x = _sample_pos_by_weights(w, rng); 
                    if pos_x is None: break
                    x = int(x_cols[pos_x])
                    d_rows, d_w = _row_neighbors_csr(self.XD, x)
                    if d_rows is None: break
                    if RW_AVOID_BACKTRACK and prev_d is not None and d_rows.size>1:
                        dw = d_w.copy(); mask = (d_rows==prev_d)
                        if mask.any(): dw[mask]=0.0
                        pos_d = _sample_pos_by_weights(dw, rng)
                    else:
                        pos_d = _sample_pos_by_weights(d_w, rng)
                    if pos_d is None: break
                    next_d = int(d_rows[pos_d])
                    if rng.random() < RW_RESTART_PROB: next_d = int(d0)
                    seq.append(next_d); prev_d, cur_d = cur_d, next_d
                    if RW_X_NO_REPEAT_LAST>0:
                        last_x.append(x); 
                        if len(last_x) > RW_X_NO_REPEAT_LAST: last_x.pop(0)
                if len(seq)>=2: yield [str(int(t)) for t in seq]

# 构造两视图的在线语料（只创建迭代器，不物化全量）
tag_corpus  = WalkCorpus(DT_ppmi, XD_tag, degX_tag, start_tag, name="tag",  base_seed=2025+11)
text_corpus = WalkCorpus(DW_bm25, XD_txt, degX_txt, start_txt, name="text", base_seed=2025+23)

print(f"[RW] tag ~{len(tag_corpus):,} sentences @L={RW_L_DOCS_PER_SENT}; text ~{len(text_corpus):,} sentences @L={RW_L_DOCS_PER_SENT}")
if RW_INSPECT_SAMPLES > 0:
    samp = []
    for i, s in enumerate(tag_corpus):
        samp.append(s); 
        if len(samp)>=RW_INSPECT_SAMPLES: break
    print("[RW] sample(tag):", samp[:min(2,len(samp))])

pd.DataFrame([{
    "RW_WALKS_PER_DOC": RW_WALKS_PER_DOC,
    "RW_L_DOCS_PER_SENT": RW_L_DOCS_PER_SENT,
    "RW_SEED_BASE": RW_SEED_BASE,
    "RW_AVOID_BACKTRACK": RW_AVOID_BACKTRACK,
    "RW_RESTART_PROB": RW_RESTART_PROB,
    "RW_X_DEGREE_POW": RW_X_DEGREE_POW,
    "RW_X_NO_REPEAT_LAST": RW_X_NO_REPEAT_LAST,
}]).to_parquet(TMP_DIR / "rw_params.parquet", engine="fastparquet", index=False)

[RW] tag ~2,145,850 sentences @L=40; text ~4,172,040 sentences @L=40


In [36]:
# —— WS-SGNS（两视图一致） ——
SGNS_DIM        = 256         # 向量维度（可扩到384，但内存↑）
SGNS_WINDOW     = 5           # 上下文窗口
SGNS_NEGATIVE   = 15          # 负采样个数（可试20）
SGNS_EPOCHS     = 8           # 训练轮次（可试10）
SGNS_SAMPLE     = 1e-5        # 高频下采样（更激进，抑制高频“噪声”上下文）
SGNS_NS_EXP     = 0.75        # 负采样分布幂次
SGNS_MIN_COUNT  = 1           # 每个doc_idx都应被建模
SGNS_BATCH_WORDS= 50_000      # 批量词数，加速吞吐
SGNS_WORKERS    = None        # None -> 使用所有可用CPU核
SGNS_SEED_TAG   = 131         # Tag视图SGNS种子
SGNS_SEED_TEXT  = 271         # Text视图SGNS种子
# 训练后是否做快速邻居抽样检查（0=不做）
SGNS_INSPECT_NN  = 0

In [39]:
# --- Step 6 · Execute: Train WS-SGNS (Tag/Text) & Extract L2-normalized Embeddings ---

# 读回核心文档与词表
doc_df    = pd.read_parquet(TMP_DIR / "doc_clean.parquet",  engine="fastparquet")
tag_vocab = pd.read_parquet(TMP_DIR /"tag_vocab.parquet",  engine="fastparquet")
text_vocab= pd.read_parquet(TMP_DIR / "text_vocab.parquet", engine="fastparquet")

# 通用：从 Parquet 三元组还原 CSR
def _load_csr_from_parquet_triplets(path, shape, dtype=np.float32):
    df = pd.read_parquet(path, engine="fastparquet")
    coo = sparse.coo_matrix((df["val"].astype(dtype), (df["row"], df["col"])),
                            shape=shape, dtype=dtype)
    return coo.tocsr()

# 读回二部图矩阵
DT_ppmi = _load_csr_from_parquet_triplets("./tmp/DT_ppmi.parquet",
                                          shape=(len(doc_df), len(tag_vocab)))
DW_bm25 = _load_csr_from_parquet_triplets("./tmp/DW_bm25.parquet",
                                          shape=(len(doc_df), len(text_vocab)))

# 读回“随机游走参数”（让 Step 6 不依赖 Step 5 的内存变量）
rw = pd.read_parquet(TMP_DIR / "rw_params.parquet", engine="fastparquet").iloc[0]
RW_WALKS_PER_DOC    = int(rw["RW_WALKS_PER_DOC"])
RW_L_DOCS_PER_SENT  = int(rw["RW_L_DOCS_PER_SENT"])
RW_SEED_BASE        = int(rw["RW_SEED_BASE"])
RW_AVOID_BACKTRACK  = bool(rw["RW_AVOID_BACKTRACK"])
RW_RESTART_PROB     = float(rw["RW_RESTART_PROB"])
RW_X_DEGREE_POW     = float(rw["RW_X_DEGREE_POW"])
RW_X_NO_REPEAT_LAST = int(rw["RW_X_NO_REPEAT_LAST"])

# 预计算与邻接
DT_ppmi = DT_ppmi.tocsr().astype(np.float32, copy=False)
DW_bm25 = DW_bm25.tocsr().astype(np.float32, copy=False)

degD_tag = np.asarray(DT_ppmi.getnnz(axis=1)).ravel()
degX_tag = np.asarray(DT_ppmi.getnnz(axis=0)).ravel()
degD_txt = np.asarray(DW_bm25.getnnz(axis=1)).ravel()
degX_txt = np.asarray(DW_bm25.getnnz(axis=0)).ravel()
XD_tag = DT_ppmi.transpose().tocsr()
XD_txt = DW_bm25.transpose().tocsr()
start_tag = np.where(degD_tag > 0)[0]
start_txt = np.where(degD_txt > 0)[0]

def _row_neighbors_csr(mat_csr, r):
    a,b = mat_csr.indptr[r], mat_csr.indptr[r+1]
    if a==b: return None, None
    return mat_csr.indices[a:b], mat_csr.data[a:b]

def _sample_pos_by_weights(weights, rng):
    if weights is None or len(weights)==0: return None
    w = np.asarray(weights, dtype=np.float64); tot = w.sum()
    if not np.isfinite(tot) or tot<=0: return None
    cdf = np.cumsum(w)
    pos = int(np.searchsorted(cdf, rng.random()*tot, side="left"))
    if pos >= cdf.size: pos = cdf.size-1
    return pos

class WalkCorpus:
    """按需生成的 D-only 语料；每次 __iter__ 都会重掷随机数并打乱起点。"""
    def __init__(self, DX, XD, degX, starts, name, base_seed):
        self.DX, self.XD, self.degX = DX, XD, degX
        self.starts = starts
        self.name = name
        self.base_seed = base_seed
        self._iters = 0
    def __len__(self):
        return int(len(self.starts) * RW_WALKS_PER_DOC)
    def __iter__(self):
        rng = np.random.default_rng(self.base_seed + 17*self._iters); self._iters += 1
        starts = self.starts.copy(); rng.shuffle(starts)
        x_deg_factor = np.power(np.maximum(self.degX,1.0), RW_X_DEGREE_POW).astype(np.float64) if abs(RW_X_DEGREE_POW)>1e-12 else None
        for si, d0 in enumerate(starts, 1):
            if RW_PROGRESS_EVERY and si % RW_PROGRESS_EVERY == 0:
                print(f"[RW-{self.name}] progressed: {si:,}/{len(starts):,}")
            for _ in range(RW_WALKS_PER_DOC):
                seq = [int(d0)]; prev_d=None; cur_d=int(d0); last_x=[]
                for _step in range(RW_L_DOCS_PER_SENT-1):
                    x_cols, x_w = _row_neighbors_csr(self.DX, cur_d)
                    if x_cols is None: break
                    w = x_w.astype(np.float64); 
                    if x_deg_factor is not None: w = w * x_deg_factor[x_cols]
                    if RW_X_NO_REPEAT_LAST>0 and last_x:
                        mask = np.isin(x_cols, last_x[-RW_X_NO_REPEAT_LAST:])
                        if mask.any(): w = w.copy(); w[mask]=0.0
                    pos_x = _sample_pos_by_weights(w, rng); 
                    if pos_x is None: break
                    x = int(x_cols[pos_x])
                    d_rows, d_w = _row_neighbors_csr(self.XD, x)
                    if d_rows is None: break
                    if RW_AVOID_BACKTRACK and prev_d is not None and d_rows.size>1:
                        dw = d_w.copy(); mask = (d_rows==prev_d)
                        if mask.any(): dw[mask]=0.0
                        pos_d = _sample_pos_by_weights(dw, rng)
                    else:
                        pos_d = _sample_pos_by_weights(d_w, rng)
                    if pos_d is None: break
                    next_d = int(d_rows[pos_d])
                    if rng.random() < RW_RESTART_PROB: next_d = int(d0)
                    seq.append(next_d); prev_d, cur_d = cur_d, next_d
                    if RW_X_NO_REPEAT_LAST>0:
                        last_x.append(x); 
                        if len(last_x) > RW_X_NO_REPEAT_LAST: last_x.pop(0)
                if len(seq)>=2: yield [str(int(t)) for t in seq]

try:
    from gensim.models import Word2Vec
except Exception as e:
    print("[ERROR] 需要 gensim>=4.3： pip install -U gensim==4.3.2"); 
    raise

if SGNS_WORKERS is None:
    SGNS_WORKERS = max(1, os.cpu_count() or 1)

# --- Tag 视图 SGNS ---
print(f"[SGNS-tag] ~{len(tag_corpus):,} sentences @L={RW_L_DOCS_PER_SENT}, dim={SGNS_DIM}, neg={SGNS_NEGATIVE}, epochs={SGNS_EPOCHS}")
tag_corpus  = WalkCorpus(DT_ppmi, XD_tag, degX_tag, start_tag, name="tag",  base_seed=RW_SEED_BASE+11)
text_corpus = WalkCorpus(DW_bm25, XD_txt, degX_txt, start_txt, name="text", base_seed=RW_SEED_BASE+23)

model_tag = Word2Vec(
    sentences    = tag_corpus,
    vector_size  = SGNS_DIM,
    window       = SGNS_WINDOW,
    sg           = 1,
    negative     = SGNS_NEGATIVE,
    hs           = 0,
    min_count    = SGNS_MIN_COUNT,
    workers      = SGNS_WORKERS,
    epochs       = SGNS_EPOCHS,
    sample       = SGNS_SAMPLE,
    ns_exponent  = SGNS_NS_EXP,
    seed         = SGNS_SEED_TAG,
    batch_words  = SGNS_BATCH_WORDS,
)

# 提取 Z_tag
N = len(doc_df)
Z_tag = np.zeros((N, SGNS_DIM), dtype=np.float32)
keys_tag = model_tag.wv.key_to_index
hit_tag = 0
for d in range(N):
    tok = str(d)
    if tok in keys_tag:
        Z_tag[d] = model_tag.wv[tok]
        hit_tag += 1
# L2归一
nz_mask = np.linalg.norm(Z_tag, axis=1) > 0
Z_tag[nz_mask] /= np.linalg.norm(Z_tag[nz_mask], axis=1, keepdims=True)

emb_tag = pd.DataFrame(Z_tag, columns=[f"f{i}" for i in range(Z_tag.shape[1])])
emb_tag.insert(0, "doc_idx", np.arange(Z_tag.shape[0], dtype=np.int64))
emb_tag.to_parquet(TMP_DIR / "Z_tag.parquet", engine="fastparquet", index=False)

print(f"[SGNS-tag] vectors built: covered_docs={hit_tag}/{N} ({hit_tag/max(N,1):.1%}), nonzero_rows={int(nz_mask.sum())}")

# --- Text 视图 SGNS ---
print(f"[SGNS-text] ~{len(text_corpus):,} sentences @L={RW_L_DOCS_PER_SENT}, dim={SGNS_DIM}, neg={SGNS_NEGATIVE}, epochs={SGNS_EPOCHS}")
model_text = Word2Vec(
    sentences    = text_corpus,
    vector_size  = SGNS_DIM,
    window       = SGNS_WINDOW,
    sg           = 1,
    negative     = SGNS_NEGATIVE,
    hs           = 0,
    min_count    = SGNS_MIN_COUNT,
    workers      = SGNS_WORKERS,
    epochs       = SGNS_EPOCHS,
    sample       = SGNS_SAMPLE,
    ns_exponent  = SGNS_NS_EXP,
    seed         = SGNS_SEED_TEXT,
    batch_words  = SGNS_BATCH_WORDS,
)

Z_text = np.zeros((N, SGNS_DIM), dtype=np.float32)
keys_txt = model_text.wv.key_to_index
hit_txt = 0
for d in range(N):
    tok = str(d)
    if tok in keys_txt:
        Z_text[d] = model_text.wv[tok]
        hit_txt += 1
nz_mask_t = np.linalg.norm(Z_text, axis=1) > 0
Z_text[nz_mask_t] /= np.linalg.norm(Z_text[nz_mask_t], axis=1, keepdims=True)

emb_txt = pd.DataFrame(Z_text, columns=[f"f{i}" for i in range(Z_text.shape[1])])
emb_txt.insert(0, "doc_idx", np.arange(Z_text.shape[0], dtype=np.int64))
emb_txt.to_parquet(TMP_DIR / "Z_text.parquet", engine="fastparquet", index=False)

print(f"[SGNS-text] vectors built: covered_docs={hit_txt}/{N} ({hit_txt/max(N,1):.1%}), nonzero_rows={int(nz_mask_t.sum())}")

# （可选）快速NN抽样
if SGNS_INSPECT_NN and hit_tag>0 and hit_txt>0:
    import random
    samp = random.sample([i for i in range(N) if str(i) in keys_tag], k=min(3, hit_tag))
    for idx in samp:
        sims = model_tag.wv.most_similar(str(idx), topn=5)
        print(f"[NN-tag] doc_idx={idx}: {[(int(t), round(float(s),4)) for t,s in sims]}")
    samp2 = random.sample([i for i in range(N) if str(i) in keys_txt], k=min(3, hit_txt))
    for idx in samp2:
        sims = model_text.wv.most_similar(str(idx), topn=5)
        print(f"[NN-text] doc_idx={idx}: {[(int(t), round(float(s),4)) for t,s in sims]}")


[SGNS-tag] ~2,145,850 sentences @L=40, dim=256, neg=15, epochs=8
[RW-tag] progressed: 50,000/214,585
[RW-tag] progressed: 100,000/214,585
[RW-tag] progressed: 150,000/214,585
[RW-tag] progressed: 200,000/214,585
[RW-tag] progressed: 50,000/214,585
[RW-tag] progressed: 100,000/214,585
[RW-tag] progressed: 150,000/214,585
[RW-tag] progressed: 200,000/214,585
[RW-tag] progressed: 50,000/214,585
[RW-tag] progressed: 100,000/214,585
[RW-tag] progressed: 150,000/214,585
[RW-tag] progressed: 200,000/214,585
[RW-tag] progressed: 50,000/214,585
[RW-tag] progressed: 100,000/214,585
[RW-tag] progressed: 150,000/214,585
[RW-tag] progressed: 200,000/214,585
[RW-tag] progressed: 50,000/214,585
[RW-tag] progressed: 100,000/214,585
[RW-tag] progressed: 150,000/214,585
[RW-tag] progressed: 200,000/214,585
[RW-tag] progressed: 50,000/214,585
[RW-tag] progressed: 100,000/214,585
[RW-tag] progressed: 150,000/214,585
[RW-tag] progressed: 200,000/214,585
[RW-tag] progressed: 50,000/214,585
[RW-tag] progress

KeyboardInterrupt: 

[RW-text] progressed: 400,000/417,204
